# Week 50, Day 2: Falsification: checking a model by simulating it

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w050/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Stay after wins and losses

Complete run(): on each trial compute utilities beta Q plus kappa for the previous choice, the probability of option 1, then either sample a choice and outcome (simulation) or score the observed choice (fitting), and update Q. Write signatures(a, rw), returning stay probability after wins and after losses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(2017)
T = 400

def run(par, sticky, a_obs=None, rw_obs=None):
    alpha, beta = par[0], par[1]; kappa = par[2] if sticky else 0.0
    Q = np.zeros(2); prev = -1; a = np.zeros(T, int); rw = np.zeros(T); L = 0.0
    for t in range(T):
        mu = np.array([0.3, 0.7]) if (t // 100) % 2 == 0 else np.array([0.7, 0.3])
        u = beta * Q + kappa * (np.arange(2) == prev)
        p1 = 1 / (1 + np.exp(-(u[1] - u[0])))
        if a_obs is None:
            a[t] = r.random() < p1; rw[t] = r.random() < mu[a[t]]
        else:
            a[t], rw[t] = a_obs[t], rw_obs[t]
            L -= np.log(max(p1 if a[t] == 1 else 1 - p1, 1e-9))
        Q[a[t]] += alpha * (rw[t] - Q[a[t]]); prev = a[t]
    return a, rw, L

def signatures(a, rw):
    ...

a, rw, _ = run([0.3, 3.0, 1.5], sticky=True)
stay_win, stay_loss = signatures(a, rw)
print(f"participant: stay after win {stay_win:.3f}, stay after loss {stay_loss:.3f}")

In [ ]:
import numpy as np
def test_stays():
    "the participant stays after a loss more than 75% of the time and after a win more than 85%"
    assert stay_win > 0.85 and stay_loss > 0.75

test_stays()
print("All checks passed")

### Exercise: Plain learning and learning with perseveration

Fit the plain model (alpha, beta) and the sticky model (alpha, beta, kappa) to the participant's choices with L-BFGS-B.

In [ ]:
fits = {}
for name, sticky, x0, bounds in [("plain", False, [0.5, 3.0], [(0.01, 1), (0, 20)]), ("sticky", True, [0.5, 3.0, 0.0], [(0.01, 1), (0, 20), (-5, 5)])]:
    fits[name] = ...
    print(f"{name:6s}: parameters {np.round(fits[name].x, 2)}, NLL {fits[name].fun:.1f}")

In [ ]:
import numpy as np
def test_better():
    "the perseveration model has a lower negative log-likelihood by more than 10"
    assert fits["plain"].fun - fits["sticky"].fun > 10
def test_beta():
    "the plain model's beta is inflated above the true 3"
    assert fits["plain"].x[1] > 3.5

test_better()
test_beta()
print("All checks passed")

### Exercise: Posterior predictive check

Simulate each fitted model 30 times with its fitted parameters, compute the two stay probabilities for each simulation, and average them.

In [ ]:
pred = {}
for name, sticky in [("plain", False), ("sticky", True)]:
    sims = [...]
    pred[name] = np.mean(sims, 0)
    print(f"{name:6s} model simulations: stay after win {pred[name][0]:.3f}, after loss {pred[name][1]:.3f}")
print(f"data:                        stay after win {stay_win:.3f}, after loss {stay_loss:.3f}")
x = np.arange(2)
plt.bar(x - 0.25, [stay_win, stay_loss], 0.25, label="data"); plt.bar(x, pred["plain"], 0.25, label="plain"); plt.bar(x + 0.25, pred["sticky"], 0.25, label="sticky")
plt.xticks(x, ["after win", "after loss"]); plt.ylabel("stay probability"); plt.ylim(0.5, 1); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_plain_fails():
    "the plain model underpredicts staying after losses by more than 0.1"
    assert stay_loss - pred["plain"][1] > 0.1
def test_sticky_ok():
    "the sticky model reproduces both signatures within 0.05"
    assert abs(pred["sticky"][0] - stay_win) < 0.05 and abs(pred["sticky"][1] - stay_loss) < 0.05

test_plain_fails()
test_sticky_ok()
print("All checks passed")